# Machine Learning for Applied Research

## 6: Metrics & Model Evaluation

In this notebook, we will cover:
- Error measures for regression: MAE, RMSE, and R²
- Meeting a new dataset: AI4I 2020 Predictive Maintenance
- Classification metrics: confusion matrices, precision, recall, and ROC/AUC
- Choosing a metric that matches your actual research question

### Introduction

Throughout the past, we compared models using RMSE without asking too many questions about it. We now start by questioning metrics themselves: what different regression metrics actually measure, why accuracy alone can be dangerously misleading for classification, and how to choose a metric that matches your actual research question rather than defaulting to whatever a tutorial happened to use.

### Error Measures for Regression: Beyond RMSE

Three metrics cover most regression needs:

- **RMSE (root mean squared error)**: squares each error before averaging, then takes the square root. Because errors are squared, large errors are penalized disproportionately more than small ones.
- **MAE (mean absolute error)**: averages the absolute value of each error directly. Every unit of error counts the same regardless of size, making MAE more robust to a few large outliers than RMSE.
- **R² (coefficient of determination)**: the proportion of variance in the target that the model explains, on a scale where 1.0 is a perfect fit, 0 means "no better than always predicting the average," and it can go negative for a genuinely bad model. Unlike RMSE and MAE, R² is scale-free, which makes it useful for comparing performance across different targets or datasets.

Which to prioritize depends on your situation: RMSE when large errors are especially costly (a "safety-critical" context where one big miss is worse than many small ones), MAE when you want errors weighted proportionally and want robustness to occasional outliers, and R² when you want a normalized sense of "how much of the pattern did we actually capture."

In [ ]:
!pip install ucimlrepo --quiet

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

energy = fetch_ucirepo(id=242)
X = energy.data.features
y = energy.data.targets

X.columns = ["Relative Compactness",
             "Surface Area",
             "Wall Area",
             "Roof Area",
             "Overall Height",
             "Orientation",
             "Glazing Area",
             "Glazing Area Distribution"]

y.columns = ["Heating Load", "Cooling Load"]

target_col = y.columns[0]   # Heating Load
df = pd.concat([X, y[[target_col]]], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=[target_col]), df[target_col], test_size=0.25, random_state=42
)

linreg = LinearRegression()
linreg.fit(X_train, y_train)

forest = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
forest.fit(X_train, y_train)

#### 1.1 Exercises:
> (a) Import `mean_absolute_error`, `mean_squared_error`, and `r2_score` from `sklearn.metrics`.
>
> (b) For both `linreg` and `forest`, compute MAE, RMSE, and R² on the test set. (RMSE is the square root of `mean_squared_error`.)
>
> (c) Do all three metrics agree on which model is better? Would you generally expect regression metrics to always agree with each other - why or why not?

In [ ]:
# Your solution:





##### Solution:


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

for name, model in [("Linear Regression", linreg), ("Random Forest", forest)]:
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse_val = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    print(f"{name:20s} - MAE: {mae:.2f} | RMSE: {rmse_val:.2f} | R2: {r2:.3f}")

# (c) In this case all three metrics likely agree on which model is
# better, since one model is dominating across the board. That won't
# always happen: MAE and RMSE can disagree when one model makes mostly
# small errors but occasionally a very large one (RMSE punishes this more
# than MAE does), so it's worth checking more than one metric rather than
# assuming they'll always point the same way.

#### 1.2 Exercises:
> (a) Take `y_test` and create a copy where you artificially inflate just the single largest actual value by adding 50 to it (simulating one bad measurement or data-entry error). Recompute MAE and RMSE for `linreg`'s predictions against this modified target.
>
> (b) Compare the new MAE and RMSE to the original values from Exercise 1.1. Which metric moved more?
>
> (c) In 1-2 sentences, explain why this happens, referring back to how each metric is defined.

In [ ]:
# Your solution:





##### Solution:


In [ ]:
y_test_outlier = y_test.copy()
outlier_idx = y_test_outlier.idxmax()
y_test_outlier.loc[outlier_idx] += 50

preds = linreg.predict(X_test)
print(f"Original       - MAE: {mean_absolute_error(y_test, preds):.2f} | RMSE: {np.sqrt(mean_squared_error(y_test, preds)):.2f}")
print(f"With 1 outlier - MAE: {mean_absolute_error(y_test_outlier, preds):.2f} | RMSE: {np.sqrt(mean_squared_error(y_test_outlier, preds)):.2f}")

# (c) RMSE moves considerably more than MAE. MAE averages absolute
# errors directly, so one bad value among ~190 test rows barely shifts
# the average. RMSE squares each error before averaging, so a single
# large error contributes disproportionately to the total.
# The "large errors penalized more" property described above, now visible
# in a single concrete number rather than just asserted.

### Meet a New Dataset: AI4I 2020 Predictive Maintenance

For classification, we switch to a new dataset: 10,000 records from a simulated manufacturing process, each describing one production run and whether the machine failed during it.

#### 1.3 Exercises:
> (a) Fetch the AI4I 2020 Predictive Maintenance dataset using `fetch_ucirepo(id=601)`.
>
> (b) Print the feature and target column names to see what is available.
>
> (c) Build a working dataframe `maint_df` using just these columns as features: `Type`, `Air temperature`, `Process temperature`, `Rotational speed [rpm]`, `Torque`, `Tool wear`, plus `Machine failure` as the target. (We are deliberately leaving out the specific failure-mode columns like `TWF`/`HDF`/`PWF`/`OSF`/`RNF` - those record *why* a failure happened and would give away the answer.)
>
> (d) Print the shape of `maint_df`, and check the class balance of the target using `.value_counts()`.

In [ ]:
# Your solution:





##### Solution:


In [ ]:
maintenance = fetch_ucirepo(id=601)
print(f"Feature columns: {maintenance.data.features.columns.tolist()}")
print(f"Target columns:  {maintenance.data.targets.columns.tolist()}")

feature_cols = ["Type", "Air temperature", "Process temperature",
                 "Rotational speed", "Torque", "Tool wear"]

maint_df = pd.concat(
    [maintenance.data.features[feature_cols], maintenance.data.targets[["Machine failure"]]],
    axis=1
)

print(f"\nShape: {maint_df.shape}")
print(f"\nClass balance:\n{maint_df['Machine failure'].value_counts()}")
print(f"\nFailure rate: {maint_df['Machine failure'].mean():.1%}")

#### Type

With roughly 3% of machines failing, keep that number in mind - it will matter a great deal for how we choose a metric later in this notebook.

`Type` (the product quality variant: L, M, or H) is categorical, and most models need numbers, not letters. We will do a quick manual conversion here since it has a natural order, and cover proper categorical encoding later.

In [ ]:
type_mapping = {"L": 0, "M": 1, "H": 2}
maint_df["Type"] = maint_df["Type"].map(type_mapping)

X_maint = maint_df.drop(columns=["Machine failure"])
y_maint = maint_df["Machine failure"]

X_maint_train, X_maint_test, y_maint_train, y_maint_test = train_test_split(
    X_maint, y_maint, test_size=0.25, random_state=42, stratify=y_maint
)

print(f"Train failure rate: {y_maint_train.mean():.1%}")
print(f"Test failure rate:  {y_maint_test.mean():.1%}")

Notice the `stratify=y_maint` argument, which we have not used before: with a rare positive class like this one, an ordinary random split could easily land you with a test set containing very few (or by bad luck, zero) failures. `stratify` forces both the train and test sets to keep roughly the same failure rate as the full dataset.

#### 1.4 Exercises:
> (a) Using `maint_df`, plot a histogram of `Torque [Nm]` for rows where `Machine failure == 0`, and a separate histogram for rows where `Machine failure == 1` (two `plt.hist(..., alpha=0.5)` calls on the same axes, with a legend, is an easy way to compare them directly).
>
> (b) Do the two distributions look meaningfully different, or largely overlapping?
>
> (c) Based on the plot, would you expect `Torque` to be a useful feature for predicting failure on its own? Why or why not?

In [ ]:
# Your solution:





##### Solution:


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))
plt.hist(maint_df.loc[maint_df["Machine failure"] == 0, "Torque"], bins=30, alpha=0.5, label="No failure")
plt.hist(maint_df.loc[maint_df["Machine failure"] == 1, "Torque"], bins=30, alpha=0.5, label="Failure")
plt.xlabel("Torque")
plt.ylabel("Count")
plt.legend()
plt.title("Torque distribution by failure status")
plt.show()

# (c) The failure-class distribution typically shows more mass toward
# the extremes (both very low and very high torque) compared to the
# no-failure distribution, which suggests Torque alone carries some
# signal, though the two distributions overlap substantially in the
# middle, so no single threshold on Torque would cleanly separate the
# classes. This is exactly the kind of per-class EDA (extending
# Notebook 2's approach to a classification target) worth doing before
# you train anything, on every dataset with a categorical target.

### Classification Metrics: Beyond Accuracy

With a classifier ready to train, we hit the same question we saw with regression: how do we measure "good"? The naive choice is accuracy - the fraction of predictions that are correct. But with only about 3% of machines failing, a model that always predicts "no failure" would already be roughly 97% accurate, while being completely useless. This is sometimes called the accuracy paradox, and it is exactly the imbalance risk you may have flagged already in Notebook 1's framing exercise about this same dataset.

A confusion matrix breaks predictions down into four counts: true positives (correctly predicted failures), false positives (predicted failure, but the machine was fine), true negatives (correctly predicted no failure), and false negatives (missed failures). From these:

- **Precision** = TP / (TP + FP): of everything we flagged as a failure, how much actually was one. High precision means few false alarms.
- **Recall** = TP / (TP + FN): of everything that actually failed, how much did we catch. High recall means few missed failures.
- **F1 score**: the harmonic mean of precision and recall, a single number balancing both.
- **ROC curve / AUC**: plots the true positive rate against the false positive rate as the decision threshold varies, and AUC summarizes that curve as a single threshold-independent number - useful when you have not yet decided exactly how aggressive the model should be.

<img src="https://drive.google.com/uc?id=1GI--TV73l8I4-0MieseF_XEgxZASQE6h" style = "height:350px"> <br>

<img src="https://drive.google.com/uc?id=18kilHE2jlCZtRXDEKMQUaBt4-2z9q3ep" style = "height:350px"> <br>

#### 1.5 Exercises:
> (a) Import the `RandomForestClassifier` from `sklearn.ensemble`.
>
> (b) Import `confusion_matrix` and `classification_report` from `sklearn.metrics`.
>
> (c) Initialize the classifier as `clf` with `random_state=42` it on the training data.
>
> (d) Use the model to `predict` the values on the test set and store the result as `y_pred`.
>
> (e) Print a confusion matrix and a classification report based on the true test values and their predictions.
>
> (f) From the confusion matrix, identify the number of true positives, false positives, true negatives, and false negatives (`confusion_matrix(...).ravel()` returns them in the order `tn, fp, fn, tp`).
>
> (g) Using those four numbers, calculate precision and recall by hand with plain arithmetic, and confirm they roughly match the values in `classification_report`'s output.
>
> (h) Import `roc_auc_score` and `roc_curve` from `sklearn.metrics`, compute the ROC AUC score using `clf.predict_proba(X_maint_test)[:, 1]`, and plot the ROC curve (you will need `import matplotlib.pyplot as plt`).
>
> (i) In your own words: if you had to choose between a model with higher precision and one with higher recall for flagging machines for maintenance, which would you prefer, and why? (Hint: consider the cost of a missed failure vs. the cost of an unnecessary inspection.)

In [ ]:
# Your solution:







##### Solution:


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

clf = RandomForestClassifier(random_state=42)
clf.fit(X_maint_train, y_maint_train)

y_pred = clf.predict(X_maint_test)
print(confusion_matrix(y_maint_test, y_pred))
print(classification_report(y_maint_test, y_pred))

tn, fp, fn, tp = confusion_matrix(y_maint_test, y_pred).ravel()
print(f"TP={tp}, FP={fp}, TN={tn}, FN={fn}")

precision_manual = tp / (tp + fp)
recall_manual = tp / (tp + fn)
print(f"\nPrecision: {precision_manual:.2f}")
print(f"Recall:    {recall_manual:.2f}")

auc = roc_auc_score(y_maint_test, clf.predict_proba(X_maint_test)[:, 1])
fpr, tpr, thresholds = roc_curve(y_maint_test, clf.predict_proba(X_maint_test)[:, 1])

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc:.2f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Machine Failure Classifier")
plt.legend()
plt.show()

# In most predictive maintenance settings, missing an actual failure
# (a false negative) is far more costly than an unnecessary inspection (a
# false positive): a missed failure can mean unplanned downtime or a
# safety incident, while an unnecessary inspection mostly costs a
# technician's time. That argues for prioritizing recall over precision,
# even at the cost of more false alarms.

#### 1.6 Exercises:
> (a) Using the `thresholds` array from `roc_curve` above (and the corresponding `tpr` array, which is the same thing as recall), find the lowest threshold value that still achieves at least 90% recall.
>
> (b) Generate predictions using that threshold instead of the default 0.5 (`(clf.predict_proba(X_maint_test)[:, 1] >= your_threshold).astype(int)`), and compute the resulting `precision_score` (imported from `metrics`).
>
> (c) Compare this precision to the precision you calculated in Exercise 1.5(b) at the default threshold. What did you give up to guarantee 90% recall?

In [ ]:
# Your solution:





##### Solution:


In [ ]:
from sklearn.metrics import precision_score

target_recall = 0.90
valid_indices = np.where(tpr >= target_recall)[0]
chosen_threshold = thresholds[valid_indices[0]]
print(f"Threshold achieving >= 90% recall: {chosen_threshold:.3f}")

y_pred_90recall = (clf.predict_proba(X_maint_test)[:, 1] >= chosen_threshold).astype(int)
precision_90recall = precision_score(y_maint_test, y_pred_90recall)
print(f"Precision at this threshold: {precision_90recall:.2f}")
print(f"Precision at default 0.5 threshold: {precision_manual:.2f}")

# (c) Precision typically drops noticeably once you force recall up to
# 90% - you catch more real failures, but at the cost of flagging more
# machines that turn out to be fine. This is the precision-recall
# trade-off made concrete: there is no free lunch, only a choice of
# where on the curve to operate, driven by what a missed failure
# actually costs versus what an unnecessary inspection costs.

### Choosing Metrics for Your Research Question

Neither regression nor classification has one "correct" metric! The right choice depends on what a wrong prediction actually costs in your context.

| Question to ask | Leans toward... |
|---|---|
| Are large errors much worse than small ones? | RMSE (regression) |
| Do you want errors weighted proportionally, robust to outliers? | MAE (regression) |
| Do you need a scale-free sense of "how much did we explain"? | R² (regression) |
| Is missing a positive case worse than a false alarm? | Recall (classification) |
| Is a false alarm more costly than a miss? | Precision (classification) |
| Need one balanced number, or to compare across many thresholds? | F1 score / ROC-AUC (classification) |

For your own research question: which of these considerations matters most, and does your intuitive choice of metric actually match it?

#### 1.7 Exercises (discussion):
> (a) If your own research produces a regression output, which of RMSE, MAE, or R² would you report as your primary metric, and why?
>
> (b) If your own research produces (or could produce) a classification output, would a false positive or a false negative be more costly in your context? What does that imply about prioritizing precision vs. recall?

In [ ]:
# Your solution:





##### Solution:

In [ ]:
# Discussion exercise - no single correct answer. A useful check: can you
# name a concrete number or scenario where your chosen metric would have
# given a different "winning" model than an alternative metric would
# have? If not, it may be worth pressure-testing your choice against a
# scenario like Exercise 1.2's outlier or 1.6's threshold shift.

#### Challenge:
> Retrain the classifier as a `LogisticRegression(max_iter=1000)` instead of `RandomForestClassifier`, and compare its ROC AUC to the Random Forest's. Which generalizes better here, and can you think of a reason connected to the fact that only about 3% of examples are failures?

In [ ]:
# Your solution:





##### Solution:


In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_maint_train, y_maint_train)

logreg_auc = roc_auc_score(y_maint_test, logreg.predict_proba(X_maint_test)[:, 1])
print(f"Logistic Regression - ROC AUC: {logreg_auc:.3f}")
print(f"Random Forest       - ROC AUC: {auc:.3f}")

# With such a rare positive class, tree-based ensembles like Random Forest
# often have an edge: repeated splits let them isolate the (often
# nonlinear) combinations of sensor readings associated with failure,
# whereas logistic regression's single linear decision boundary can
# struggle to separate a small, not-necessarily-linearly-separable
# minority class from the rest. Results vary by dataset though!
# It's worth checking both on any new imbalanced problem rather than assuming
# one family always wins.

#### Challenge 2:
> Compute the F1 score for `clf`'s default (0.5-threshold) predictions and for the 90%-recall predictions from Exercise 1.6. Which has the higher F1? (Import `f1_score`.) Does that mean the 90%-recall threshold was a bad choice or does it depend on something F1 doesn't capture?

In [ ]:
# Your solution:





##### Solution:


In [ ]:
from sklearn.metrics import f1_score

f1_default = f1_score(y_maint_test, y_pred)
f1_90recall = f1_score(y_maint_test, y_pred_90recall)

print(f"F1 at default threshold:  {f1_default:.3f}")
print(f"F1 at 90%-recall threshold: {f1_90recall:.3f}")

# The default threshold often has a higher F1 score, since F1 balances
# precision and recall equally, but that doesn't make the 90%-recall
# threshold a bad choice. F1 assumes precision and recall matter the
# same amount, which is rarely true in a real predictive-maintenance
# decision. The right threshold depends on the actual costs involved
# (Exercise 1.7), not on maximizing a metric that assumes those costs
# are symmetric.

## Conclusion

This notebook moved past "just use RMSE" and looked at evaluation on its own terms:

- **Regression metrics**: RMSE, MAE, and R² each emphasize something different about what "good" means, and now you have seen concretely, not just in theory, how an outlier moves RMSE more than MAE
- **A new dataset**: AI4I 2020 Predictive Maintenance, with a realistically rare failure rate (~3%)
- **Classification EDA**: comparing feature distributions across classes before modeling
- **Classification metrics**: confusion matrices, precision, recall, F1, and ROC/AUC, and why accuracy alone would have been misleading here
- **Threshold selection**: choosing an operating point on the precision-recall trade-off is itself a decision, not something a single metric makes for you
- **Matching metric to question**: the right metric depends on what a wrong prediction costs, not on convention